# Shared-core smoke test

**Purpose.** Run one fixed governance case through `corrected_public_engine_v1_1` in **replay_mode** (five non-compensatory gates plus compensatory scoring, without SCM abstention overrides).

**What this validates.** The checked-out engine module imports cleanly, evaluates a case end-to-end, and produces deterministic artifacts consumed by the harness (`reproduce_all.py` → manifest → validation).

**Outputs (contract).** This notebook is solely responsible for:
- `outputs/tables/smoke_test_results.csv`
- `outputs/figures/smoke_test_summary.txt`


In [1]:
from __future__ import annotations

import csv
import sys
from pathlib import Path


def repo_root() -> Path:
    start = Path.cwd().resolve()
    for p in (start, *start.parents):
        if (p / "engine" / "corrected_public_engine_v1_1.py").is_file():
            return p
    raise RuntimeError(
        "Repository root not found (expected engine/corrected_public_engine_v1_1.py)."
    )


ROOT = repo_root()
sys.path.insert(0, str(ROOT / "engine"))
import corrected_public_engine_v1_1 as eng

OUT_TAB = ROOT / "outputs" / "tables"
OUT_FIG = ROOT / "outputs" / "figures"
OUT_TAB.mkdir(parents=True, exist_ok=True)
OUT_FIG.mkdir(parents=True, exist_ok=True)

CASE = {
    "case_id": "smoke_core_001",
    "features": {
        "intrinsic_safety": 0.58,
        "evidence_strength": 0.55,
        "bias_harm_index": 0.44,
        "uncertainty_calibration": 0.52,
        "traceability_integrity": 0.53,
    },
}

result = eng.evaluate_case(CASE, profile_name="moderate", mode=eng.MODE_REPLAY)
result_hash = eng.hash_output(result)

csv_path = OUT_TAB / "smoke_test_results.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(
        [
            "case_id",
            "profile_name",
            "mode",
            "governance_outcome",
            "all_gates_pass",
            "compensatory_score",
            "compensatory_approved",
            "approved",
            "result_sha256",
        ]
    )
    w.writerow(
        [
            result["case_id"],
            result["profile_name"],
            result["mode"],
            result["governance_outcome"],
            result["all_gates_pass"],
            result["compensatory_score"],
            result["compensatory_approved"],
            result["approved"],
            result_hash,
        ]
    )

summary_path = OUT_FIG / "smoke_test_summary.txt"
lines = [
    "Shared-core smoke test (01_smoke_test)",
    "========================================",
    "",
    "Purpose: This notebook runs one fixed synthetic governance case through the",
    "corrected public engine v1.1 in replay_mode (five gates plus compensatory",
    "scoring only, matching the static historical replay path).",
    "",
    "Why this matters: A successful run proves the checked-out engine module",
    "executes end-to-end and that we can serialize a deterministic record under",
    "config/expected_outputs.json for automated manifest validation.",
    "",
    "Outputs produced (this notebook only):",
    "- outputs/tables/smoke_test_results.csv — tabular record for the case.",
    "- outputs/figures/smoke_test_summary.txt — this narrative summary.",
    "",
    f"Governance outcome:                  {result['governance_outcome']}",
    f"All gates pass (replay):             {result['all_gates_pass']}",
    f"Compensatory approved:               {result['compensatory_approved']}",
    f"Final approved flag (replay):       {result['approved']}",
    f"SHA-256 (canonical JSON result):    {result_hash}",
    "",
]
summary_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("Smoke test artifacts written.")


Smoke test artifacts written.
